In [1]:
from datasets import load_dataset
dataset = load_dataset("opus_books", "en-fr")
train_data = dataset["train"]

INFO: PyTorch version 2.0.0 available.
INFO: TensorFlow version 2.18.0 available.


In [2]:
from torchtext.data.utils import get_tokenizer
en_tokenizer = get_tokenizer("basic_english")
fr_tokenizer = get_tokenizer("basic_english")  # Approximation for French

In [3]:
from torchtext.vocab import build_vocab_from_iterator
en_vocab = build_vocab_from_iterator(
    map(en_tokenizer, [pair["translation"]["en"] for pair in train_data]),
    specials=["<unk>", "<pad>", "<sos>", "<eos>"]
)
fr_vocab = build_vocab_from_iterator(
    map(fr_tokenizer, [pair["translation"]["fr"] for pair in train_data]),
    specials=["<unk>", "<pad>", "<sos>", "<eos>"]
)
en_vocab.set_default_index(en_vocab["<unk>"])
fr_vocab.set_default_index(fr_vocab["<unk>"])

In [4]:
import torch
def preprocess_sentence(sentence, tokenizer, vocab, max_len=50):
    tokens = ["<sos>"] + tokenizer(sentence.lower()) + ["<eos>"]
    indices = [vocab[token] for token in tokens]
    if len(indices) < max_len:
        indices += [vocab["<pad>"]] * (max_len - len(indices))
    return torch.tensor(indices[:max_len])

In [5]:
## Encoder
import torch.nn as nn
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, batch_first=True)
    
    def forward(self, src):
        embedded = self.embedding(src)  # [batch_size, seq_len, emb_dim]
        outputs, (hidden, cell) = self.rnn(embedded)
        return outputs, hidden, cell  # outputs for attention

In [6]:
## Attention 
class Attention(nn.Module):
    def __init__(self, hid_dim):
        super().__init__()
        self.attn = nn.Linear(hid_dim * 2, hid_dim)
        self.v = nn.Linear(hid_dim, 1, bias=False)
    
    def forward(self, hidden, encoder_outputs):
        batch_size = encoder_outputs.shape[0]
        src_len = encoder_outputs.shape[1]
        hidden = hidden[-1].unsqueeze(1).repeat(1, src_len, 1)  # [batch, seq_len, hid_dim]
        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))
        attention = self.v(energy).squeeze(2)  # [batch, seq_len]
        return torch.softmax(attention, dim=1)

In [7]:
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim):
        super().__init__()
        self.output_dim = output_dim
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim + hid_dim, hid_dim, batch_first=True)
        self.fc = nn.Linear(hid_dim, output_dim)
        self.attention = Attention(hid_dim)
    
    def forward(self, input, hidden, cell, encoder_outputs):
        embedded = self.embedding(input.unsqueeze(1))  # [batch, 1, emb_dim]
        a = self.attention(hidden, encoder_outputs)  # [batch, seq_len]
        context = torch.bmm(a.unsqueeze(1), encoder_outputs)  # [batch, 1, hid_dim]
        rnn_input = torch.cat((embedded, context), dim=2)
        output, (hidden, cell) = self.rnn(rnn_input, (hidden, cell))
        prediction = self.fc(output.squeeze(1))
        return prediction, hidden, cell

In [8]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
    
    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = src.shape[0]
        trg_len = trg.shape[1]
        trg_vocab_size = self.decoder.output_dim
        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size)
        encoder_outputs, hidden, cell = self.encoder(src)
        input = trg[:, 0]  # <sos>
        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell, encoder_outputs)
            outputs[:, t] = output
            top1 = output.argmax(1)
            input = trg[:, t] if torch.rand(1).item() < teacher_forcing_ratio else top1
        return outputs

In [11]:
import torch

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [2]:
def train(model, iterator, optimizer, criterion, epochs=10):
    model.train()
    for epoch in range(epochs):
        epoch_loss = 0
        for batch_idx, (src, trg) in enumerate(iterator):
            print(f"Batch {batch_idx}, src device: {src.device}, trg device: {trg.device}")
            optimizer.zero_grad()
            output = model(src, trg)
            output_dim = output.shape[-1]
            output = output[:, 1:].reshape(-1, output_dim)
            trg = trg[:, 1:].reshape(-1)
            loss = criterion(output, trg)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            if batch_idx % 100 == 0:
                print(f'Batch {batch_idx}, Loss: {loss.item()}')
        avg_loss = epoch_loss / len(iterator)
        print(f"Epoch {epoch+1}, Average Loss: {avg_loss}")
    torch.save(model.state_dict(), "model.pt")


# Initialize models
en_fr_encoder = Encoder(len(en_vocab), 256, 512)
en_fr_decoder = Decoder(len(fr_vocab), 256, 512)
en_fr_model = Seq2Seq(en_fr_encoder, en_fr_decoder)

fr_en_encoder = Encoder(len(fr_vocab), 256, 512)
fr_en_decoder = Decoder(len(en_vocab), 256, 512)
fr_en_model = Seq2Seq(fr_en_encoder, fr_en_decoder)

# Optimizers and loss
optimizer_en_fr = torch.optim.Adam(en_fr_model.parameters())
optimizer_fr_en = torch.optim.Adam(fr_en_model.parameters())
criterion = nn.CrossEntropyLoss(ignore_index=en_vocab["<pad>"])

# DataLoader with corrected collate_fn
from torch.utils.data import DataLoader
def collate_fn(batch):
    src = torch.stack([preprocess_sentence(pair["translation"]["en"], en_tokenizer, en_vocab) for pair in batch])
    trg = torch.stack([preprocess_sentence(pair["translation"]["fr"], fr_tokenizer, fr_vocab) for pair in batch])
    return src, trg

# Train English-to-French model
train_loader = DataLoader(train_data, batch_size=32, collate_fn=collate_fn)
train(en_fr_model, train_loader, optimizer_en_fr, criterion)

# Train French-to-English model (swap src and trg)
train_loader = DataLoader(train_data, batch_size=32, collate_fn=lambda x: (collate_fn(x)[1], collate_fn(x)[0]))
train(fr_en_model, train_loader, optimizer_fr_en, criterion)

NameError: name 'Encoder' is not defined

In [1]:
import pandas as pd
transcription_df = pd.read_csv("final_evaluated_dataset.csv")  # Adjust path

def translate(model, sentence, src_tokenizer, trg_vocab, max_len=50):
    model.eval()
    tokens = ["<sos>"] + src_tokenizer(sentence.lower()) + ["<eos>"]
    src_indices = [trg_vocab[token] for token in tokens]
    src_tensor = torch.tensor(src_indices[:max_len]).unsqueeze(0)
    with torch.no_grad():
        encoder_outputs, hidden, cell = model.encoder(src_tensor)
        trg_indices = [trg_vocab["<sos>"]]
        for _ in range(max_len):
            input = torch.tensor([trg_indices[-1]])
            output, hidden, cell = model.decoder(input, hidden, cell, encoder_outputs)
            pred_token = output.argmax(1).item()
            trg_indices.append(pred_token)
            if pred_token == trg_vocab["<eos>"]:
                break
    return " ".join([trg_vocab.get_itos()[i] for i in trg_indices[1:-1]])

# Round-translation
for idx, row in transcription_df.iterrows():
    en_text = row["Translation"]
    fr_text = translate(en_fr_model, en_text, en_tokenizer, fr_vocab)
    en_back = translate(fr_en_model, fr_text, fr_tokenizer, en_vocab)
    transcription_df.at[idx, "French"] = fr_text
    transcription_df.at[idx, "English_Back"] = en_back

transcription_df.to_csv("round_translation.csv", index=False)

NameError: name 'en_fr_model' is not defined

## training

In [12]:
pip uninstall torch 



SyntaxError: invalid syntax (2832320480.py, line 1)

In [3]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator
from datasets import load_dataset

# Device setup
num_gpus = torch.cuda.device_count()
device = torch.device("cuda" if torch.cuda.is_available() and num_gpus > 0 else "cpu")
print(f"Using device: {device}")

# Load dataset
dataset = load_dataset("opus_books", "en-fr")
train_data = dataset["train"]

# Tokenizers
en_tokenizer = get_tokenizer("basic_english")
fr_tokenizer = get_tokenizer("basic_english")  # Approximation for French

# Build vocabularies
def build_vocab(data, tokenizer):
    return build_vocab_from_iterator(map(tokenizer, data), specials=["<unk>", "<pad>", "<sos>", "<eos>"])

en_vocab = build_vocab([pair["translation"]["en"] for pair in train_data], en_tokenizer)
fr_vocab = build_vocab([pair["translation"]["fr"] for pair in train_data], fr_tokenizer)
en_vocab.set_default_index(en_vocab["<unk>"])
fr_vocab.set_default_index(fr_vocab["<unk>"])

# Preprocessing function
def preprocess_sentence(sentence, tokenizer, vocab, max_len=50):
    tokens = ["<sos>"] + tokenizer(sentence.lower()) + ["<eos>"]
    indices = [vocab[token] for token in tokens]
    indices = indices[:max_len] + [vocab["<pad>"]] * max(0, max_len - len(indices))
    return torch.tensor(indices, device=device)

# Collate function
def collate_fn(batch):
    src = torch.stack([preprocess_sentence(pair["translation"]["en"], en_tokenizer, en_vocab) for pair in batch])
    trg = torch.stack([preprocess_sentence(pair["translation"]["fr"], fr_tokenizer, fr_vocab) for pair in batch])
    return src, trg

# Model definitions
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, batch_first=True)
    
    def forward(self, src):
        embedded = self.embedding(src)
        outputs, (hidden, cell) = self.rnn(embedded)
        return outputs, hidden, cell

class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, batch_first=True)
        self.fc = nn.Linear(hid_dim, output_dim)
    
    def forward(self, input, hidden, cell):
        embedded = self.embedding(input.unsqueeze(1))
        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
        prediction = self.fc(output.squeeze(1))
        return prediction, hidden, cell

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
    
    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size, trg_len = trg.shape
        trg_vocab_size = self.decoder.fc.out_features
        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size, device=device)
        encoder_outputs, hidden, cell = self.encoder(src)
        input = trg[:, 0]
        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell)
            outputs[:, t] = output
            top1 = output.argmax(1)
            input = trg[:, t] if torch.rand(1).item() < teacher_forcing_ratio else top1
        return outputs

# Training function
def train(model, iterator, optimizer, criterion, epochs=1, is_fr_en=False):
    model.train()
    for epoch in range(epochs):
        epoch_loss = 0
        for batch_idx, (src, trg) in enumerate(iterator):
            # Swap src and trg for French-to-English model
            if is_fr_en:
                src, trg = trg, src
            optimizer.zero_grad()
            output = model(src, trg)
            output_dim = output.shape[-1]
            output = output[:, 1:].reshape(-1, output_dim)
            trg = trg[:, 1:].reshape(-1)
            loss = criterion(output, trg)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            if batch_idx % 100 == 0:
                print(f"Epoch {epoch+1}, Batch {batch_idx}, Loss: {epoch_loss / (batch_idx + 1):.4f}")
        print(f"Epoch {epoch+1} completed, Loss: {epoch_loss:.4f}")

# Initialize models
en_fr_model = Seq2Seq(Encoder(len(en_vocab), 256, 512), Decoder(len(fr_vocab), 256, 512)).to(device)
fr_en_model = Seq2Seq(Encoder(len(fr_vocab), 256, 512), Decoder(len(en_vocab), 256, 512)).to(device)

# Define optimizers and loss functions
optimizer_en_fr = torch.optim.Adam(en_fr_model.parameters())
optimizer_fr_en = torch.optim.Adam(fr_en_model.parameters())
criterion_en_fr = nn.CrossEntropyLoss(ignore_index=fr_vocab["<pad>"])
criterion_fr_en = nn.CrossEntropyLoss(ignore_index=en_vocab["<pad>"])

# Create DataLoader
train_loader = DataLoader(train_data, batch_size=32 * max(1, num_gpus), collate_fn=collate_fn)

# Train and save models
print("Training English-to-French model...")
train(en_fr_model, train_loader, optimizer_en_fr, criterion_en_fr, epochs=10, is_fr_en=False)
torch.save(en_fr_model.state_dict(), "en_fr_model.pt")

print("Training French-to-English model...")
train(fr_en_model, train_loader, optimizer_fr_en, criterion_fr_en, epochs=10, is_fr_en=True)
torch.save(fr_en_model.state_dict(), "fr_en_model.pt")

print("Training completed!")


INFO: PyTorch version 2.2.0+cu118 available.
INFO: TensorFlow version 2.18.0 available.


Using device: cuda


README.md:   0%|          | 0.00/28.1k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/127085 [00:00<?, ? examples/s]

Training English-to-French model...
Epoch 1, Batch 0, Loss: 11.3196
Epoch 1, Batch 100, Loss: 7.2643
Epoch 1, Batch 200, Loss: 6.9949
Epoch 1, Batch 300, Loss: 6.8515
Epoch 1, Batch 400, Loss: 6.6574
Epoch 1, Batch 500, Loss: 6.5145
Epoch 1, Batch 600, Loss: 6.4295
Epoch 1, Batch 700, Loss: 6.4787
Epoch 1, Batch 800, Loss: 6.4749
Epoch 1, Batch 900, Loss: 6.4594
Epoch 1, Batch 1000, Loss: 6.4167
Epoch 1, Batch 1100, Loss: 6.3554
Epoch 1, Batch 1200, Loss: 6.2909
Epoch 1, Batch 1300, Loss: 6.2154
Epoch 1, Batch 1400, Loss: 6.1509
Epoch 1, Batch 1500, Loss: 6.0941
Epoch 1, Batch 1600, Loss: 6.0897
Epoch 1, Batch 1700, Loss: 6.0738
Epoch 1, Batch 1800, Loss: 6.0899
Epoch 1, Batch 1900, Loss: 6.0867
Epoch 1, Batch 2000, Loss: 6.0690
Epoch 1, Batch 2100, Loss: 6.0880
Epoch 1, Batch 2200, Loss: 6.1002
Epoch 1, Batch 2300, Loss: 6.1095
Epoch 1, Batch 2400, Loss: 6.1000
Epoch 1, Batch 2500, Loss: 6.1038
Epoch 1, Batch 2600, Loss: 6.0940
Epoch 1, Batch 2700, Loss: 6.0846
Epoch 1, Batch 2800, Lo

KeyboardInterrupt: 

In [10]:
import torch
import torch.nn as nn

class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, batch_first=True)
    
    def forward(self, src):
        embedded = self.embedding(src)
        outputs, (hidden, cell) = self.rnn(embedded)
        return outputs, hidden, cell

class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, batch_first=True)
        self.fc = nn.Linear(hid_dim, output_dim)
    
    def forward(self, input, hidden, cell):
        embedded = self.embedding(input.unsqueeze(1))
        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
        prediction = self.fc(output.squeeze(1))
        return prediction, hidden, cell

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
    
    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size, trg_len = trg.shape
        trg_vocab_size = self.decoder.fc.out_features
        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size, device=src.device)
        encoder_outputs, hidden, cell = self.encoder(src)
        input = trg[:, 0]
        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell)
            outputs[:, t] = output
            top1 = output.argmax(1)
            input = trg[:, t] if torch.rand(1).item() < teacher_forcing_ratio else top1
        return outputs

In [11]:
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator
from datasets import load_dataset

# Load dataset
dataset = load_dataset("opus_books", "en-fr")
train_data = dataset["train"]

# Tokenizers
en_tokenizer = get_tokenizer("basic_english")
fr_tokenizer = get_tokenizer("basic_english")

# Build vocabularies
def build_vocab(data, tokenizer):
    return build_vocab_from_iterator(map(tokenizer, data), specials=["<unk>", "<pad>", "<sos>", "<eos>"])

en_vocab = build_vocab([pair["translation"]["en"] for pair in train_data], en_tokenizer)
fr_vocab = build_vocab([pair["translation"]["fr"] for pair in train_data], fr_tokenizer)
en_vocab.set_default_index(en_vocab["<unk>"])
fr_vocab.set_default_index(fr_vocab["<unk>"])

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# English-to-French model
en_fr_encoder = Encoder(input_dim=len(en_vocab), emb_dim=256, hid_dim=512)
en_fr_decoder = Decoder(output_dim=len(fr_vocab), emb_dim=256, hid_dim=512)
en_fr_model = Seq2Seq(en_fr_encoder, en_fr_decoder).to(device)
en_fr_model.load_state_dict(torch.load("en_fr_model.pt", map_location=device))
en_fr_model.eval()

# French-to-English model
fr_en_encoder = Encoder(input_dim=len(fr_vocab), emb_dim=256, hid_dim=512)
fr_en_decoder = Decoder(output_dim=len(en_vocab), emb_dim=256, hid_dim=512)
fr_en_model = Seq2Seq(fr_en_encoder, fr_en_decoder).to(device)
fr_en_model.load_state_dict(torch.load("fr_en_model.pt", map_location=device))
fr_en_model.eval()

Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(82152, 256)
    (rnn): LSTM(256, 512, batch_first=True)
  )
  (decoder): Decoder(
    (embedding): Embedding(68480, 256)
    (rnn): LSTM(256, 512, batch_first=True)
    (fc): Linear(in_features=512, out_features=68480, bias=True)
  )
)

In [13]:
def translate_sentence(sentence, model, tokenizer, src_vocab, trg_vocab, device, max_len=50):
    model.eval()
    # Preprocess input
    tokens = ["<sos>"] + tokenizer(sentence.lower()) + ["<eos>"]
    src_tensor = torch.tensor([src_vocab[token] for token in tokens], device=device).unsqueeze(0)
    
    # Encode
    with torch.no_grad():
        _, hidden, cell = model.encoder(src_tensor)  # Ignore encoder_outputs
    
    # Decode
    trg_indexes = [trg_vocab["<sos>"]]
    for _ in range(max_len):
        trg_tensor = torch.tensor([trg_indexes[-1]], device=device)
        with torch.no_grad():
            output, hidden, cell = model.decoder(trg_tensor, hidden, cell)
        pred_token = output.argmax(1).item()
        trg_indexes.append(pred_token)
        if pred_token == trg_vocab["<eos>"]:
            break
    
    # Convert indices to words
    trg_tokens = [trg_vocab.get_itos()[i] for i in trg_indexes][1:-1]  # Skip <sos> and <eos>
    return " ".join(trg_tokens)

# Example usage
sentence = "Hello, how are you?"
translation = translate_sentence(sentence, en_fr_model, en_tokenizer, en_vocab, fr_vocab, device)
print(f"English to French: {translation}")

sentence = "Bonjour, comment vas-tu ?"
translation = translate_sentence(sentence, fr_en_model, fr_tokenizer, fr_vocab, en_vocab, device)
print(f"French to English: {translation}")

English to French: ah ! tu es ' as pas .
French to English: well , you , ?


## Task 7

In [2]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'  # For CUDA debugging if needed

import torch
import torch.nn as nn
import pandas as pd
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator
from datasets import load_dataset

# Set up device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load training dataset to rebuild vocabularies (same as used during training)
dataset = load_dataset("opus_books", "en-fr")
train_data = dataset["train"]

# Initialize tokenizers
en_tokenizer = get_tokenizer("basic_english")
fr_tokenizer = get_tokenizer("basic_english")

# Function to build vocabularies from training data
def build_vocab(data, tokenizer):
    return build_vocab_from_iterator(map(tokenizer, data), specials=["<unk>", "<pad>", "<sos>", "<eos>"])

# Build English and French vocabularies
en_vocab = build_vocab([pair["translation"]["en"] for pair in train_data], en_tokenizer)
fr_vocab = build_vocab([pair["translation"]["fr"] for pair in train_data], fr_tokenizer)
en_vocab.set_default_index(en_vocab["<unk>"])
fr_vocab.set_default_index(fr_vocab["<unk>"])

# Define the Encoder class
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, batch_first=True)
    
    def forward(self, src):
        embedded = self.embedding(src)
        outputs, (hidden, cell) = self.rnn(embedded)
        return outputs, hidden, cell

# Define the Decoder class
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, batch_first=True)
        self.fc = nn.Linear(hid_dim, output_dim)
    
    def forward(self, input, hidden, cell):
        embedded = self.embedding(input.unsqueeze(1))
        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
        prediction = self.fc(output.squeeze(1))
        return prediction, hidden, cell

# Define the Seq2Seq model
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
    
    def forward(self, src, trg=None, teacher_forcing_ratio=0.5):
        if trg is None:  # Inference mode
            batch_size = src.shape[0]
            max_len = 50  # Maximum length for inference
            trg_vocab_size = self.decoder.fc.out_features
            outputs = torch.zeros(batch_size, max_len, trg_vocab_size).to(src.device)
            encoder_outputs, hidden, cell = self.encoder(src)
            input = torch.tensor([self.decoder.embedding.num_embeddings - 2], device=src.device)  # <sos>
            for t in range(max_len):
                output, hidden, cell = self.decoder(input, hidden, cell)
                outputs[:, t, :] = output
                input = output.argmax(1)
            return outputs
        else:  # Training mode (not used here)
            encoder_outputs, hidden, cell = self.encoder(src)
            outputs = torch.zeros(trg.shape[0], trg.shape[1], self.decoder.fc.out_features).to(src.device)
            input = trg[:, 0]
            for t in range(1, trg.shape[1]):
                output, hidden, cell = self.decoder(input, hidden, cell)
                outputs[:, t, :] = output
                input = trg[:, t] if torch.rand(1).item() < teacher_forcing_ratio else output.argmax(1)
            return outputs

# Initialize and load the models
en_fr_model = Seq2Seq(Encoder(len(en_vocab), 256, 512), Decoder(len(fr_vocab), 256, 512)).to(device)
fr_en_model = Seq2Seq(Encoder(len(fr_vocab), 256, 512), Decoder(len(en_vocab), 256, 512)).to(device)
en_fr_model.load_state_dict(torch.load("en_fr_model.pt", map_location=device))
fr_en_model.load_state_dict(torch.load("fr_en_model.pt", map_location=device))
en_fr_model.eval()
fr_en_model.eval()

# Base translation function
def translate_sentence(sentence, model, tokenizer, src_vocab, trg_vocab, device, max_len=50):
    model.eval()
    tokens = ["<sos>"] + tokenizer(sentence.lower()) + ["<eos>"]
    src_tensor = torch.tensor([src_vocab[token] for token in tokens], device=device).unsqueeze(0)
    with torch.no_grad():
        _, hidden, cell = model.encoder(src_tensor)
    trg_indexes = [trg_vocab["<sos>"]]
    for _ in range(max_len):
        trg_tensor = torch.tensor([trg_indexes[-1]], device=device)
        with torch.no_grad():
            output, hidden, cell = model.decoder(trg_tensor, hidden, cell)
        pred_token = output.argmax(1).item()
        trg_indexes.append(pred_token)
        if pred_token == trg_vocab["<eos>"]:
            break
    trg_tokens = [trg_vocab.get_itos()[i] for i in trg_indexes][1:-1]  # Exclude <sos> and <eos>
    return " ".join(trg_tokens)

# Safe translation function to handle non-string inputs
def safe_translate(sentence, model, tokenizer, src_vocab, trg_vocab, device, max_len=50):
    if isinstance(sentence, str):  # Check if input is a string
        return translate_sentence(sentence, model, tokenizer, src_vocab, trg_vocab, device, max_len)
    else:
        return "No translation available"  # Placeholder for invalid inputs (e.g., NaN)

# Function for round-trip translation
def round_trip_translate(sentence, en_fr_model, fr_en_model, en_tokenizer, fr_tokenizer, en_vocab, fr_vocab, device):
    french_translation = safe_translate(sentence, en_fr_model, en_tokenizer, en_vocab, fr_vocab, device)
    if french_translation != "No translation available":
        back_to_english = safe_translate(french_translation, fr_en_model, fr_tokenizer, fr_vocab, en_vocab, device)
    else:
        back_to_english = "No translation available"
    return french_translation, back_to_english

# Load the CSV file
df = pd.read_csv("final_evaluated_dataset.csv")

# Check column names to confirm "Translation" (adjust if necessary)
print("Columns in CSV:", df.columns.tolist())
column_name = "Translation"  # Adjust to "Traslation" if that’s the actual name in your CSV

# Perform round-trip translation for each sentence
results = []
for sentence in df[column_name]:
    if pd.notna(sentence):  # Check if the value is not NaN
        french, back_to_english = round_trip_translate(sentence, en_fr_model, fr_en_model, en_tokenizer, fr_tokenizer, en_vocab, fr_vocab, device)
        results.append({"original": sentence, "french": french, "back_to_english": back_to_english})
    else:
        results.append({"original": "No translation available", "french": "No translation available", "back_to_english": "No translation available"})

# Save results to a new CSV
results_df = pd.DataFrame(results)
results_df.to_csv("round_trip_translations.csv", index=False)
print("Round-trip translations saved to 'round_trip_translations.csv'")

Using device: cuda
Columns in CSV: ['Start Time', 'End Time', 'Sentence', 'Corrected Sentence', 'Translation', 'Student_Emotion', 'Pipeline_Emotion', 'Core_Pipeline_Emotion', 'Corrected_Emotion']
Round-trip translations saved to 'round_trip_translations.csv'


In [3]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'  # For CUDA debugging if needed

import torch
import torch.nn as nn
import pandas as pd
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator
from datasets import load_dataset

# Set up device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load training dataset to rebuild vocabularies (same as used during training)
dataset = load_dataset("opus_books", "en-fr")
train_data = dataset["train"]

# Initialize tokenizers
en_tokenizer = get_tokenizer("basic_english")
fr_tokenizer = get_tokenizer("basic_english")

# Function to build vocabularies from training data
def build_vocab(data, tokenizer):
    return build_vocab_from_iterator(map(tokenizer, data), specials=["<unk>", "<pad>", "<sos>", "<eos>"])

# Build English and French vocabularies
en_vocab = build_vocab([pair["translation"]["en"] for pair in train_data], en_tokenizer)
fr_vocab = build_vocab([pair["translation"]["fr"] for pair in train_data], fr_tokenizer)
en_vocab.set_default_index(en_vocab["<unk>"])
fr_vocab.set_default_index(fr_vocab["<unk>"])

# Define the Encoder class
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, batch_first=True)
    
    def forward(self, src):
        embedded = self.embedding(src)
        outputs, (hidden, cell) = self.rnn(embedded)
        return outputs, hidden, cell

# Define the Decoder class
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, batch_first=True)
        self.fc = nn.Linear(hid_dim, output_dim)
    
    def forward(self, input, hidden, cell):
        embedded = self.embedding(input.unsqueeze(1))
        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
        prediction = self.fc(output.squeeze(1))
        return prediction, hidden, cell

# Define the Seq2Seq model
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
    
    def forward(self, src, trg=None, teacher_forcing_ratio=0.5):
        if trg is None:  # Inference mode
            batch_size = src.shape[0]
            max_len = 50  # Maximum length for inference
            trg_vocab_size = self.decoder.fc.out_features
            outputs = torch.zeros(batch_size, max_len, trg_vocab_size).to(src.device)
            encoder_outputs, hidden, cell = self.encoder(src)
            input = torch.tensor([self.decoder.embedding.num_embeddings - 2], device=src.device)  # <sos>
            for t in range(max_len):
                output, hidden, cell = self.decoder(input, hidden, cell)
                outputs[:, t, :] = output
                input = output.argmax(1)
            return outputs
        else:  # Training mode (not used here)
            encoder_outputs, hidden, cell = self.encoder(src)
            outputs = torch.zeros(trg.shape[0], trg.shape[1], self.decoder.fc.out_features).to(src.device)
            input = trg[:, 0]
            for t in range(1, trg.shape[1]):
                output, hidden, cell = self.decoder(input, hidden, cell)
                outputs[:, t, :] = output
                input = trg[:, t] if torch.rand(1).item() < teacher_forcing_ratio else output.argmax(1)
            return outputs

# Initialize and load the models
en_fr_model = Seq2Seq(Encoder(len(en_vocab), 256, 512), Decoder(len(fr_vocab), 256, 512)).to(device)
fr_en_model = Seq2Seq(Encoder(len(fr_vocab), 256, 512), Decoder(len(en_vocab), 256, 512)).to(device)
en_fr_model.load_state_dict(torch.load("en_fr_model.pt", map_location=device))
fr_en_model.load_state_dict(torch.load("fr_en_model.pt", map_location=device))
en_fr_model.eval()
fr_en_model.eval()

# Base translation function
def translate_sentence(sentence, model, tokenizer, src_vocab, trg_vocab, device, max_len=50):
    model.eval()
    tokens = ["<sos>"] + tokenizer(sentence.lower()) + ["<eos>"]
    src_tensor = torch.tensor([src_vocab[token] for token in tokens], device=device).unsqueeze(0)
    with torch.no_grad():
        _, hidden, cell = model.encoder(src_tensor)
    trg_indexes = [trg_vocab["<sos>"]]
    for _ in range(max_len):
        trg_tensor = torch.tensor([trg_indexes[-1]], device=device)
        with torch.no_grad():
            output, hidden, cell = model.decoder(trg_tensor, hidden, cell)
        pred_token = output.argmax(1).item()
        trg_indexes.append(pred_token)
        if pred_token == trg_vocab["<eos>"]:
            break
    trg_tokens = [trg_vocab.get_itos()[i] for i in trg_indexes][1:-1]  # Exclude <sos> and <eos>
    return " ".join(trg_tokens)

# Safe translation function to handle non-string inputs
def safe_translate(sentence, model, tokenizer, src_vocab, trg_vocab, device, max_len=50):
    if isinstance(sentence, str):  # Check if input is a string
        return translate_sentence(sentence, model, tokenizer, src_vocab, trg_vocab, device, max_len)
    else:
        return "No translation available"  # Placeholder for invalid inputs (e.g., NaN)

# Function for round-trip translation
def round_trip_translate(sentence, en_fr_model, fr_en_model, en_tokenizer, fr_tokenizer, en_vocab, fr_vocab, device):
    french_translation = safe_translate(sentence, en_fr_model, en_tokenizer, en_vocab, fr_vocab, device)
    if french_translation != "No translation available":
        back_to_english = safe_translate(french_translation, fr_en_model, fr_tokenizer, fr_vocab, en_vocab, device)
    else:
        back_to_english = "No translation available"
    return french_translation, back_to_english

# Load the CSV file
df = pd.read_csv("processed_group 9_url1.csv")

# Check column names to confirm "Translation" (adjust if necessary)
print("Columns in CSV:", df.columns.tolist())
column_name = "Translation"  # Adjust to "Traslation" if that’s the actual name in your CSV

# Perform round-trip translation for each sentence
results = []
for sentence in df[column_name]:
    if pd.notna(sentence):  # Check if the value is not NaN
        french, back_to_english = round_trip_translate(sentence, en_fr_model, fr_en_model, en_tokenizer, fr_tokenizer, en_vocab, fr_vocab, device)
        results.append({"original": sentence, "french": french, "back_to_english": back_to_english})
    else:
        results.append({"original": "No translation available", "french": "No translation available", "back_to_english": "No translation available"})

# Save results to a new CSV
results_df = pd.DataFrame(results)
results_df.to_csv("round_trip_translations.csv", index=False)
print("Round-trip translations saved to 'round_trip_translations.csv'")

Using device: cuda
The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
Columns in CSV: ['Start Time', 'End Time', 'Sentence', 'Corrected Sentence', 'Translation', 'Emotion', 'Intensity']
Round-trip translations saved to 'round_trip_translations.csv'
